In [ ]:
#   numpy(넘파이)는 숫자 여러 개를 '배열(array)'로 묶어 한 번에 계산하게 해 주는 라이브러리입니다.
#   예) [160, 170, 180]에 모두 2를 더하는 계산을 한 줄로 할 수 있습니다.
#   이 노트북에서는 데이터 준비/정규화 같은 'NumPy 흐름'에 사용합니다.
import numpy as np

#   torch는 PyTorch 라이브러리입니다.
#   이번 노트북에서는 다음을 쓰기 위해 사용합니다.
#       (1) torch.nn.Linear : H(x)=a·X_norm + b 계산을 대신해 주는 부품
#       (2) torch.nn.BCELoss: Binary Cross Entropy Cost 계산을 대시해 주는 부품
#       (3) 자동 미분(autograd)과 optimizer(torch.optim.SGD)
#
#   참고: 이 노트북에서는 'import torch.nn as nn' 같은 줄임 import를 쓰지 않습니다.
#       항상 torch.nn.Linear, torch.nn.BCELoss 처럼 전체 이름을 그대로 적습니다.
#       그래야 이 기능들이 torch.nn 안에 있는 API라는 사실이 코드에 그대로 보입니다.
import torch

In [ ]:
#   ========================================
#   1. 입력값 X와 정답 y 준비   (이전 파일과 동일)
#   ========================================

#   X: 입력값입니다. 여기서는 사람의 키(cm)를 사용합니다.
#       np.array([...]) 는 여러 숫자를 하나의 NumPy 배열로 묶는 것입니다.
X = np.array([160, 170, 180, 190])

#   y: 정답값입니다. 0은 농구선수 아님, 1은 농구선수입니다.
#       X와 y는 순서대로 짝지어집니다. 즉 키 160 → 정답 0, 키 190 → 정답 1.
y = np.array([0, 0, 1, 1])

print('입력값 X: ', X)
print('입력값 y: ', y)

입력값 X:  [160 170 180 190]
입력값 y:  [0 0 1 1]


In [ ]:
#   ========================================
#   2. 입력값 정규화    (이전 파일과 동일)
#   ========================================

#   평균(mean)과 표준편차(std)를 계산합니다.
#   - 평균      : 데이터의 중심(가운데쯤 되는 값)
#   - 표준편차   : 데이터가 평균에서 얼마나 넓게 퍼져 있는지를 나타내는 값
X_mean = np.mean(X)
X_std = np.std(X)

#   정규화 공식: (원본값 - 평균) / 표준편차
#   주의: 실제 학습에는 원래 키 X가 아니라, 정규하된 입력값 X_norm을 사용합니다.
#         (X_mean, X_std는 나중에 '새 입력값 예측'에서도 똑같이 다시 씁니다.)
X_norm = (X - X_mean) / X_std

print('입력값 평균 X_mean: ', X_mean)
print('입력값 표준편차 X_std: ', X_std)
print('정규화된 입력값 X_norm: ', X_norm)

입력값 평균 X_mean:  175.0
입력값 표준편차 X_std:  11.180339887498949
정규화된 입력값 X_norm:  [-1.34164079 -0.4472136   0.4472136   1.34164079]


In [ ]:
#   ========================================
#   2-1. X_norm 과 t를 PyTorch tensor 로 변환하고 shape을 (n, 1)로 정리
#   ========================================

#   학습에 사용할 입력값(X_norm)과 정답(y)을 tensor로 바꿔 둡니다.
#
#   주의(햇갈리기 쉬움):
#       X       : 원래 키(cm)
#       X_norm  : 정규화된 입력값 ← 학습에는 이 값을 사용합니다.
#   따라서 아래에서도 X가 아니라 X_norm을 tensor로 변환합니다.

#   dtype=torch.float32 : 소수점 계산(미분)을 위해 실수(float) 형식으로 만듭니다.
X_norm_tensor = torch.tensor(X_norm, dtype = torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32)

#   torch.nn.Linear(1, 1)에 넣으려면 각 데이터가 '입력 특성 1개'를 가진 형태,
#   즉 shape (n, 1) 이어야 합니다. 그래서 reshape(-1, 1)로 모양을 바꿉니다.
#       -1 : 행 개수는 알아서 (여기서는 4)
#        1 : 열 개수는 1 (입력 특성 1개)
X_norm_tensor = X_norm_tensor.reshape(-1, 1)
y_tensor = y_tensor.reshape(-1, 1)

print('학습용 입력 tensor X_norm_tensor:\n', X_norm_tensor)
print('학습용 정답 tensor y_tensor:\n', y_tensor)

#   shape을 꼭 확인합니다. 둘 다 (4, 1) 이어야 합니다.
print('X_norm_tensor shape: ', X_norm_tensor.shape)
print('y_tensor shape: ', y_tensor.shape)

학습용 입력 tensor X_norm_tensor:
 tensor([[-1.3416],
        [-0.4472],
        [ 0.4472],
        [ 1.3416]])
학습용 정답 tensor y_tensor:
 tensor([[0.],
        [0.],
        [1.],
        [1.]])
X_norm_tensor shape:  torch.Size([4, 1])
y_tensor shape:  torch.Size([4, 1])


In [ ]:
#   ========================================
#   3. torch.nn.Linear(1, 1) 생성   (기존 a, b 를 직접 만들지 않습니다.)
#   ========================================

#   torch.manual_seed(42)
#       - PyTorch의 랜덤 초기값을 고정합니다.
#       - nn.Linear는 weight(=a)외 bias(=b)를 랜덤으로 초기화하므로,
#         seed를 고정해야 실행할 때마다 결과가 크게 달라지지 않습니다.
#       - 반드시 linear를 만들기 '직전'에 둡니다.
torch.manual_seed(42)

#   torch.nn.Linear(1, 1)
#       - 입력 특성 1개(키 하나)를 받아 출력 1개(H(x) 값 하나)를 내보내는 부품입니다.
#       - 내부적으로 H(x) = a·X_norm + b를 계산합니다.
#       - 기존 강의의 a는 linear.weight, 기존 강의의 b는 linear.bias 입니다.
#       - 이전 파일처럼 a, b tensor를 직접 만들지 않는다는 점이 핵심입니다.
linear = torch.nn.Linear(1, 1)

#   linear 안에 자동으로 ㅁ나들어진 weight(=a)와 bias(=b)를 확인합니다.
#   학생들이 'a, b가 사라졌다'고 느끼지 않도록 학습 '전'의 초기값을 직접 봐 둡니다.
print('linear.weight: ', linear.weight)#   기존 강의의 a 역할
print('linear.bias: ', linear.bias)#   기존 강의의 b 역할

linear.weight:  Parameter containing:
tensor([[0.7645]], requires_grad=True)
linear.bias:  Parameter containing:
tensor([0.8300], requires_grad=True)


In [ ]:
#   ========================================
#   4. torch.nn.BCELoss() 생성  (직접 BCE 수식을 적지 않습니다)
#   ========================================

#   torch.nn,BCELoss()
#       - Binary Cross Entropy Cost를 계산해 주는 PyTorch 부품입니다.
#       - 사용법: mean_cost = criterion(z, y_tensor)
#       - 주의: 입력으로 H가 아니라, sigmoid를 통과한 예측 확률 z를 넣어야 합니다!
#               (H를 넣는 BCEWithKogitsLoss는 이번 노트북에서는 사용하지 않습니다.)
#       - 변수 이름은 관례적으로 criterion(판정 기준)이라고 짓습니다.
criterion = torch.nn.BCELoss()

print('criterion 준비 완료: ', criterion)

criterion 준비 완료:  BCELoss()


In [ ]:
#   ========================================
#   5. 학습 설정    (learning_rate, epochs)
#   ========================================

#   learning_rate(학습률): 한 번에 weight, bias를 얼마나 크게 수정할지 정하는 값입니다.
#   이 값은 바로 다름 셀에서 optimizer를 만들 대 lr=learning_rate 로 넘겨줍니다.
learning_rate = 0.1

#   epochs(에폭): 전체 데이터를 몇 번 반복해서 학습할지 정하는 값입니다.
#   여기서는 같은 데이터로 1000번 반복 학습합니다.
epochs = 1000

print('learning_rate: ', learning_rate)
print('epochs: ', epochs)

learning_rate:  0.1
epochs:  1000


In [ ]:
#   ========================================
#   6. optimizer 생성   (linear.parameters() 를 넘깁니다)
#   ========================================

#   torch.optim.SGD(linear.parameters(), lr=learning_rate)
#       - linear.parameters():  optimizer가 업데이트할 학습 대상입니다.
#                               linear 안의 weight(=a)와 bias(=b)를 가져옵니다.
#       - lr=learning_rate   : 한 번에 얼마나 움직일지(학습률). 위에서 정한 0.1을 넘겨줍니다.
#
#   이전 파일에서는 [a, b]를 직접 넘겼지만, 이번에는 a, b가 linear 안에 있으므로
#   linear.parameters()를 넘긴다는 점이 핵심입니다.
optimizer = torch.optim.SGD(linear.parameters(), lr = learning_rate)

#   linear.parameters()가 실제로 무엇을 가져오는지 눈으로 확인합니다.
#   이 출결 결과에 보이는 값들이 optimizer가 업데이트할 학습 대상입니다.
#   (첫 번째가 weight(=a), 두 번쨰가 bias(=b)입니다.)
print(list(linear.parameters()))

[Parameter containing:
tensor([[0.7645]], requires_grad=True), Parameter containing:
tensor([0.8300], requires_grad=True)]


In [ ]:
#   ========================================
#   7. nn.Linear + nn.BECLoss 로 경사 하강법 학습   (이번 실습의 핵심 루프)
#   ========================================
#
#   한 번의 epoch에서 일어나는 단계 (반드시 이 순서):
#       1. optimizer.zero_grad()        : 이전 epoch의 grad를 0으로 초기화
#       2. H = linear(X_norm_tensor)    : torch.nn.Linear 로 H(x) = a·X_norm + b 계산
#       3. z = torch.sigmoid(H)         : 예측 확률
#       4. mean_cost = criterion(z, y)  : torch.nn.BCELoss로 Cost 계산  (z를 넣음! H 아님)
#       5. mean_cost.backward()         : linear.weight.grad, linear.bias.grad 자동 계산
#       6. optimizer.step()             : linear.weight, linear.bias 업데이트
#       7. 학습 상태 출력

for epoch in range(epochs):
    
    #   ----- 1. 이전 epoch에서 계산된 grad 초기화 -----
    #   PyTorch의 grad는 덮어쓰기가 아니라 '누적(더하기)'됩니다.
    #   optimizer는 linear의 weight, bias를 관리하므로,
    #   이 한 줄이 linear.weight.grad, linear.bias.grad 를 한꺼번에 0으로 만듭니다.
    optimizer.zero_grad()
    
    #   ----- 2. H(x) = a·X_norm + b 계산 -----
    #   linear(X_norm_tensor)를 실행하면 PyTorch가 내부적으로 H(x) = a·X_norm + b 를 계산합니다.
    #   이때 a는 linear.weight, b는 linear.bias 입니다.
    #   H(x)는 sigmoid에 들어가기 전의 선형 계산값입니다. (확률이 아닙니다.)
    H = linear(X_norm_tensor)
    
    #   ----- 3. H(x) = a·X_norm + b 계산 -----
    #   z는 0~1 사이의 예측 확률입니다.
    z = torch.sigmoid(H)
    
    #   ----- 4. torch.nn.BCELoss 로 Cost 계산 -----
    #   주의: criterion에는 H가 아니라, sigmoid를 통과한 z를 넣습니다!
    #        (H를 넣은 것은 나중에 배울 BCEWithLogitsLoss 방식이며, 이번 파일에서는 쓰지 않습니다.)
    mean_cost = criterion(z, y_tensor)
        
    #   ----- 5. backward: linear.weight.grad, linear.bias.grad 자동 계산 -----
    #   mean_cost 에서 출발해 weight, bias까지 거꾸로 따라가며 미분값을 구해
    #       linear.weight.grad  (=  기존 grad_a)
    #       linear.bias.grad    (=  기존 grad_b)
    #   에 저장합니다.
    mean_cost.backward()
    
    #   ----- 6. optimizer가 weight와 bias 업데이트 -----
    #   optimizer.step()은 linear.weight.grad, linear.bias.grad를 사용해
    #   Cost가 줄어드는 방향으로 linear.weight, linear.bias를 수정합니다.
    optimizer.step()
    
    #   ----- 7. 학습 상태 출력 -----
    #   입력 특성이 1개라 weight, bias에 값이 하나씩만 있으므로, .item()으로 숫자만 꺼냅니다.
    #   100 epoch마다 한 번씩, 그리고 마지막 epoch에서 출력합니다.
    if epoch % 100 == 0 or epoch == epochs - 1:
        print(
            f'epoch = {epoch},  '
            f'Cost = {mean_cost.item():.6f},  '
            f'weight(a) = {linear.weight.item():.6f},  '
            f'bias(b) = {linear.bias.item():.6f}'
        )
        
    #   (참고) 초반 3 epoch에서만 grad 값을 확인해 봅니다.
    #   기존 autograd 실습에서 a.grad, b.grad 를 확인하던 것을
    #   이번에는 linear.weight.grad, linear.bias.grad 로 확인합니다.
    #       기존 grad_a → linear.weight.grad
    #       기존 grad_b → linear.bias.grad
    if epoch < 3:
        print(
            f'  (확인용) linear.weight.grad = {linear.weight.grad.item():.6f},  '
            f'linear.bias.grad  =   {linear.bias.grad.item():.6f}'
            
        )

epoch = 0,  Cost = 0.495464,  weight(a) = 0.793780,  bias(b) = 0.812529
  (확인용) linear.weight.grad = -0.292415,  linear.bias.grad  =   0.174793
  (확인용) linear.weight.grad = -0.286153,  linear.bias.grad  =   0.169918
  (확인용) linear.weight.grad = -0.280072,  linear.bias.grad  =   0.165171
epoch = 100,  Cost = 0.178670,  weight(a) = 2.290082,  bias(b) = 0.173212
epoch = 200,  Cost = 0.125357,  weight(a) = 3.002210,  bias(b) = 0.061586
epoch = 300,  Cost = 0.099283,  weight(a) = 3.509002,  bias(b) = 0.026837
epoch = 400,  Cost = 0.082901,  weight(a) = 3.912263,  bias(b) = 0.013229
epoch = 500,  Cost = 0.071398,  weight(a) = 4.250606,  bias(b) = 0.007116
epoch = 600,  Cost = 0.062789,  weight(a) = 4.543496,  bias(b) = 0.004091
epoch = 700,  Cost = 0.056068,  weight(a) = 4.802371,  bias(b) = 0.002480
epoch = 800,  Cost = 0.050660,  weight(a) = 5.034644,  bias(b) = 0.001570
epoch = 900,  Cost = 0.046207,  weight(a) = 5.245449,  bias(b) = 0.001031
epoch = 999,  Cost = 0.042507,  weight(a) = 5.

In [ ]:
#   ========================================
#   8. 학습 완료 후 최종 weight(a), bias(b) 확인
#   ========================================

#   학습된 weight, bias는 optimizer.step()에 의해 1000번 반복 업데이트된 값입니다.
#   (정규화된 입력값 X_norm을 기준으로 학습된 값이라는 점을 기억하세요.)
#
#   입력 특성이 1개라 값이 하나씩만 있으므로 .item()으로 숫자만 꺼냅니다.
print('학습된 weight(a): ', linear.weight.item())
print('학습된 bias(b): ', linear.bias.item())

#   tensor 원본 형태도 함께 확인해 둡니다. (shape과 requires_grad 표시를 볼 수 있습니다.)
print('linear.weight: ', linear.weight)
print('linear.bias: ', linear.bias)

학습된 weight(a):  5.436656951904297
학습된 bias(b):  0.0007013267604634166
linear.weight:  Parameter containing:
tensor([[5.4367]], requires_grad=True)
linear.bias:  Parameter containing:
tensor([0.0007], requires_grad=True)


In [ ]:
#   ========================================
#   9. 새로운 입력값 예측
#   ========================================

#   키가 185cm인 사람이 농구선수인지 예측합니다.
input_height = 185

#   새로운 입력값도 학습 데이터와 '같은 기준'으로 정규화해야 합니다.
#   학습 때 사용한 X_mean, X_std를 그대로 다시 사용합니다.  (새로 계산하면 안 됩니다.)
input_norm = (input_height - X_mean) / X_std

#   예측은 학습이 아니므로 weight, bias를 업데이트하지 않습니다.
#   따라서 미분 계산 기록도 필요 없으므로 with torch.no_grad() 안에서 계산합니다.
with torch.no_grad():
    #   torch.nn.Linear(1, 1)에 넣으려면 입력 shape을 (1, 1)로 맞춰야 합니다.
    #       [[input_norm]] : 이중 대괄호로 감싸 (데이터 1개, 입력 특성 1개) = (1, 1) 형태로 만듭니다.
    input_norm_tensor = torch.tensor([[input_norm]], dtype = torch.float32)
    print('input_norm_tensor shape:', input_norm_tensor.shape)
    
    #   H(x) = a·X_norm + b (학습된 linear가 계산 = 선형 계산값이며 확률이 아닙니다.)
    H_new = linear(input_norm_tensor)
    #   z = sigmoid(H)  (예측 확률 - 0~1 사이)
    z_new = torch.sigmoid(H_new)
    #   0.5 이상이면 1(농구선수), 미만이면 0(농구선수 아님)
    #   z_new는 shape (1, 1) tensor이므로 .item()으로 숫자 하나를 꺼냅니다.
    pred = 1 if z_new.item() >= 0.5 else 0
    
print(f'키가 {input_height}cm인 사람의 농구선수 확률(z): {z_new.item():.4f}')
if pred == 1:
    print('판별 결과: 농구선수입니다.')
else:
    print('판별 결과: 농구선수가 아닙니다.')

##  자습용 체크리스트1
- torch.nn.Linear(1,1)은 어떤 계산을 대신 하는가?:
    H(x) = a·X_norm_tensor + b 계산을 대신 함
- linear.weight는 기존 강의의 무엇에 해당하는가?
    H(x) 회귀식 a의 역할음 함
- linear.bias는 기존 강의의 무엇에 해당하는가?
    H(x) 회귀식 b의 역할을 함

##   자습용 체크리스트2
- z는 왜 확률이라고 부를 수 있는가?
    이진분류 진행을 위해 회귀식 H(x)에 대해 시그모이드 처리 진행해야 함(출력 값의 범위도 0~1)
- BCELoss에는 H와 z중 무엇을 넣어야 하는가?
    선형 회귀식 H가 아닌 확률 z을 넣어야 함
- Linear가 sigmoid까지 자동으로 계산해주는가?
    torch.nn.Linear() 통해 회귀식 생성 후, torch.sigmoid를 통해 별도로 시그모이드 z를 생성해야 함

##  코드 흐름 확인 1
- 왜 X_norm_tensor의 shape을 (n, 1)로 바꾸는가?
    기존 행렬 형태에 대해서는 torch.nn.Linear() 함수를 통해 선형 회귀식 생성이 불가능 함
- optimizer에 [a, b] 대신 linear.parameters()를 넘기는 이유는 무엇인가?
    기존 수동으로 생성하던 a·X_norm_tensor + b 대신, torch.nn.Linear()를 통해 생성된 weight 및 bias에 대해서 optimizer 과정을 진행해야 함
- optimizer.zero_grad()는 왜 매 epoch마다 필요한가?
    grad는 매 학습마다 축척되기 때문에 초기화 과정이 필요함

##  코드 흐름 확인 2
- Cost.gackward() 후 어떤 grad가 계산되는가?
    mean_cost에서 weight, bias까지 미분값을 연산함
- optimizer.step()은 어떤 값을 업데이트하는가?
    Cost가 줄어드는 방향으로 linear.weight, linear.bias를 수정함
- Cost.item()은 왜 출력할 때 사용하는가?
    숫자값만을 확인하기 위하여 사용함

##  예측 흐름 확인
- 새 입력값을 예측할 떄도 왜 정규화가 필요한가?
    기존에 정규화가 진행된 데이터를 통해 학습을 진행하였기 때문에 새로운 데이터에 대해서도 정규화를 진행하여 분석을 진행해야 함
- 예측할 때 torch.no_grad()를 사용하는 이유는 무엇인가?
    weight, bias에 대하여 업데이트를 진행할 필요가 없기 때문에 사용함
- 새 입력값 185cm는 왜 shape을 (1, 1)로 바꾸어야 하는가?
    해당 데이터에 대해서도 torch.nn.Linear()를 통해 회귀식을 구성해야 함